# ds006465 (3M-CPSEED) dataset audit

This notebook only reads BIDS metadata, event tables, and optionally one EDF header. It does **not** preprocess or overwrite EEG data. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
from IPython.display import display

def locate_dataset(start: Path) -> Path:
    for parent in (start, *start.parents):
        candidate = parent / 'data' / 'ds006465'
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not locate data/ds006465. Open this notebook inside eeg_to_voice_chinese.')

DATASET = locate_dataset(Path.cwd().resolve())
print(f'DATASET = {DATASET}')
print(json.loads((DATASET / 'dataset_description.json').read_text())['Name'])

## 1. BIDS file inventory

The EDF paths may be git-annex symlinks, but listing them does not trigger a download.

In [ ]:
edf_files = sorted(DATASET.glob('sub-*/ses-*/eeg/*_eeg.edf'))
event_files = sorted(DATASET.glob('sub-*/ses-*/eeg/*_events.tsv'))
channel_files = sorted(DATASET.glob('sub-*/ses-*/eeg/*_channels.tsv'))
sidecar_files = sorted(DATASET.glob('sub-*/ses-*/eeg/*_eeg.json'))
subjects = sorted({path.parts[-4] for path in event_files})
sessions = sorted({path.parts[-3] for path in event_files}, key=lambda x: int(x.split('-')[1]))

summary = pd.Series({
    'subjects': len(subjects),
    'sessions_per_subject': len(sessions),
    'raw EDF recordings': len(edf_files),
    'events.tsv files': len(event_files),
    'channels.tsv files': len(channel_files),
    'EEG sidecar JSON files': len(sidecar_files),
})
display(summary.to_frame('count'))
print('Subjects:', ', '.join(subjects))
print('Sessions:', ', '.join(sessions))

## 2. Trial and label audit from BIDS events

A row in `events.tsv` is treated as one labelled event/trial marker. This is the reproducible count for the BIDS raw layer.

In [ ]:
event_frames = []
for path in event_files:
    match = re.search(r'(sub-\d+)/(ses-\d+)/eeg/', path.as_posix())
    frame = pd.read_csv(path, sep='\t')
    frame['subject'] = match.group(1)
    frame['session'] = match.group(2)
    frame['event_file'] = path.relative_to(DATASET).as_posix()
    event_frames.append(frame)

events = pd.concat(event_frames, ignore_index=True)
events['trial_type'] = events['trial_type'].astype(str)
label_counts = events['trial_type'].value_counts().sort_index(key=lambda s: s.astype(int))

print(f'BIDS event trials: {len(events):,}')
print(f'Distinct labels: {events.trial_type.nunique()}')
display(label_counts.rename_axis('label_code').to_frame('n_trials'))
display(events.groupby(['subject', 'session']).size().rename('n_trials').unstack(fill_value=0))

## 3. Pinyin label protocol

The README defines ten target sounds: four finals (`a`, `i`, `u`, `ü`) and six initials (`m`, `f`, `j`, `l`, `k`, `ch`). The BIDS `events.tsv` files store only codes `1`–`10`; this dataset copy does not document the code-to-Pinyin mapping. Do not assume the order below is verified until it is checked against the collection code/protocol.

In [ ]:
candidate_protocol_order = ['a', 'i', 'u', 'ü', 'm', 'f', 'j', 'l', 'k', 'ch']
label_audit = pd.DataFrame({
    'event_code': sorted(events.trial_type.astype(int).unique()),
    'candidate_protocol_order__UNVERIFIED': candidate_protocol_order,
})
display(label_audit)
print('Action required before phoneme-level modelling: verify the event-code mapping independently.')

## 4. Channel, sampling-rate, and recording-duration audit

In [ ]:
channel_rows = []
for path in channel_files:
    match = re.search(r'(sub-\d+)/(ses-\d+)/eeg/', path.as_posix())
    channels = pd.read_csv(path, sep='\t')
    channel_rows.append({
        'subject': match.group(1), 'session': match.group(2),
        'n_channels': len(channels),
        'sampling_hz': channels['sampling_frequency'].dropna().unique().tolist(),
    })
channels_audit = pd.DataFrame(channel_rows)
display(channels_audit.groupby('n_channels').size().rename('recordings').to_frame())
display(channels_audit.explode('sampling_hz').groupby('sampling_hz').size().rename('recordings').to_frame())

sidecar_rows = []
for path in sidecar_files:
    match = re.search(r'(sub-\d+)/(ses-\d+)/eeg/', path.as_posix())
    meta = json.loads(path.read_text())
    sidecar_rows.append({
        'subject': match.group(1), 'session': match.group(2),
        'sampling_hz': meta.get('SamplingFrequency'),
        'duration_s': meta.get('RecordingDuration'),
        'eeg_channels': meta.get('EEGChannelCount'),
    })
sidecars = pd.DataFrame(sidecar_rows)
display(sidecars.describe(include='all'))

recording_keys = {(p.parts[-4], p.parts[-3]) for p in edf_files}
channel_keys = set(zip(channels_audit.subject, channels_audit.session))
print('Recordings without channels.tsv:', sorted(recording_keys - channel_keys))

## 5. Optional: inspect a single EDF header

This reads only the header (`preload=False`) and leaves the raw signal unchanged.

In [ ]:
import mne

example_edf = edf_files[0]
if not example_edf.exists():
    print(f'EDF content is unavailable locally: {example_edf}')
    print('Retrieve only this recording with: datalad get ' + str(example_edf.relative_to(DATASET)))
else:
    raw = mne.io.read_raw_edf(example_edf, preload=False, verbose='ERROR')
    display(pd.Series({
        'file': example_edf.relative_to(DATASET).as_posix(),
        'sampling_hz': raw.info['sfreq'],
        'channels': len(raw.ch_names),
        'duration_s': raw.times[-1],
        'channel_names': ', '.join(raw.ch_names[:12]) + (' ...' if len(raw.ch_names) > 12 else ''),
    }).to_frame('value'))

## 6. Derivative inventory and interpretation warnings

`derivatives/preproc` contains phase-specific MATLAB files (`speak`, `intend`, `imagine`). Keep it separate from raw BIDS EDF analyses until its variables and preprocessing provenance have been verified.

In [ ]:
mat_files = sorted((DATASET / 'derivatives' / 'preproc').rglob('*.mat'))
def phase_of(path):
    name = path.name.lower()
    for phase in ('speak', 'intend', 'imagine'):
        if phase in name:
            return phase
    return 'other'

phase_counts = pd.Series([phase_of(path) for path in mat_files]).value_counts().rename_axis('phase').to_frame('n_mat_files')
display(phase_counts)

print('Key audit conclusion: raw BIDS events provide 4,000 labelled markers (20 subjects × 4 sessions × 50).')
print('The README statement of 1,600 trials per participant is not consistent with those raw event tables; resolve this before reporting a trial count beyond the BIDS marker count.')
print('The event files label codes 1–10, but do not identify whether a marker corresponds to speaking, silent articulation, or imagery.')